In [1]:
# ===============================
# STARTER SCRAPING READY
# ===============================

# Modules à importer
import pandas as pd
import numpy as np

# Options Pandas
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option("display.max_colwidth", None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_columns', 200)

In [2]:
df_base_2020 = pd.read_csv("../data/raw/resultats_municipales_2020.csv", encoding="utf-8-sig", sep=";", low_memory=False)

In [5]:
df_base_2020 = df_base_2020.rename(columns={"Code de la commune": "code_commune"})

df_base_2020["code_commune"] = df_base_2020["code_commune"].astype(str).str.zfill(3)

df_base_2020["Code du département"] = df_base_2020["Code du département"].astype(str).str.zfill(2)

df_base_2020["code_insee"] = df_base_2020["Code du département"] + df_base_2020["code_commune"]

In [6]:
df_info_region = pd.read_csv("../data/raw/population_communes.csv", encoding="utf-8-sig", sep=";")

In [7]:
df_info_region = df_info_region.rename(columns={"Code commune": "code_commune"})

In [8]:
df_info_region["code_commune"] = df_info_region["code_commune"].astype(str).str.zfill(3)

df_info_region["Code département"] = df_info_region["Code département"].astype(str).str.zfill(2)

df_info_region["code_insee"] = df_info_region["Code département"] + df_info_region["code_commune"]


In [10]:
df_base_2020_regions = pd.merge(df_base_2020, df_info_region, on="code_insee", how="inner")

In [11]:
resultats = []

for index, row in df_base_2020_regions.iterrows():
    commune = row["Libellé de la commune"]
    code_insee = row["code_insee"]
    region = row["Nom de la région"]
    participation = row["% Vot/Ins"]

    for n in range (1,17):
        nuance = row.iloc[19 + (n-1) * 12]
        pourcentage_voix = row.iloc[29 + (n-1) * 12]

        if pd.notna(nuance):
            resultats.append({
                "commune": commune,
                "code_insee": code_insee,
                "region" : region,
                "participation" : participation,
                "nuance" : nuance,
                "pourcentage_voix" : pourcentage_voix,
            })

df_long_toutes_regions = pd.DataFrame(resultats)

In [12]:
df_long_toutes_regions["nuance"] = df_long_toutes_regions["nuance"].str[1:]

In [13]:
mapping_politique_2020 = {
    # extrême-gauche
    "EXG" : "Extrême gauche",

    # gauche
    "COM" : "Gauche", "FI" : "Gauche", "SOC" : "Gauche", "RDG" : "Gauche", "DVG" : "Gauche", "UG" : "Gauche", "VEC" : "Gauche",

    # centre
    "REM" :  "Centre", "MDM" : "Centre", "UDI" : "Centre", "UC" : "Centre", "DVC" : "Centre",

    # droite
    "LR" : "Droite", "UD" : "Droite", "DVD" : "Droite", "DLF" : "Droite",

    # extrême-droite
    "RN" : "Extrême droite", "EXD" : "Extrême droite",

    # divers ou non classé
    "ECO" : "Divers ou non classé", "DIV" : "Divers ou non classé", "REG" : "Divers ou non classé", "GJ" : "Divers ou non classé", "" : "Divers ou non classé", "NC" : "Divers ou non classé"
}

In [14]:
df_long_toutes_regions["bloc"] = df_long_toutes_regions["nuance"].map(mapping_politique_2020)

In [15]:
df_long_toutes_regions["pourcentage_voix"] = df_long_toutes_regions["pourcentage_voix"].str.replace(",",".")
df_long_toutes_regions["pourcentage_voix"] = pd.to_numeric(df_long_toutes_regions["pourcentage_voix"])

In [16]:
resultats_commune_2020_bloc = df_long_toutes_regions.groupby(["commune", "code_insee", "region", "participation", "bloc"])["pourcentage_voix"].sum().reset_index()

In [17]:
resultats_commune_2020_bloc_wide = resultats_commune_2020_bloc.pivot_table(
    index=["commune", "code_insee", "region", "participation"], 
    columns="bloc", 
    values="pourcentage_voix"
).reset_index()

In [18]:
resultats_commune_2020_bloc_wide = resultats_commune_2020_bloc_wide[["commune", "code_insee", "region", "participation", "Extrême gauche", "Gauche", "Centre", "Droite", "Extrême droite", "Divers ou non classé"]]

In [19]:
resultats_commune_2020_bloc_wide["Bloc dominant"] = resultats_commune_2020_bloc_wide[["Extrême gauche", "Gauche", "Centre", "Droite", "Extrême droite", "Divers ou non classé"]].idxmax(axis=1)

In [20]:
resultats_commune_2020_nuance = df_long_toutes_regions.groupby(["code_insee", "nuance"])["pourcentage_voix"].sum().reset_index()

In [21]:
resultats_commune_2020_nuance_wide = resultats_commune_2020_nuance.pivot_table(
    index=["code_insee"], 
    columns="nuance", 
    values="pourcentage_voix"
).reset_index()

In [22]:
resultats_communes_2020_bloc_et_nuance= pd.merge(resultats_commune_2020_bloc_wide, resultats_commune_2020_nuance_wide, on="code_insee", how="inner")

In [23]:
resultats_communes_2020_bloc_et_nuance = resultats_communes_2020_bloc_et_nuance[["commune", "code_insee", "region", "participation", "Bloc dominant", "Extrême gauche", "Gauche", "Centre", "Droite", "Extrême droite", "Divers ou non classé", "EXG", "COM", "FI", "SOC", "RDG", "DVG", "UG", "VEC", "REM", "MDM", "UDI", "UC", "DVC", "NC", "ECO", "DIV", "REG", "GJ", "LR", "UD", "DVD", "DLF", "RN", "EXD"]]

In [24]:
resultats_communes_2020_bloc_et_nuance = resultats_communes_2020_bloc_et_nuance.rename(columns={
    "COM" : "Parti communiste français",
    "DIV" : "Liste(s) divers",
    "DLF" : "Debout la France",
    "DVC" : "Liste(s) divers centre",
    "DVD" : "Liste(s) divers droite",
    "DVG" : "Liste(s) divers gauche",
    "ECO" : "Autre(s) liste(s) écologiste(s)",
    "EXD" : "Liste(s) d'extrême droite",
    "EXG" : "Liste(s) d'extrême gauche",
    "FI" : "La France insoumise",
    "GJ" : "Liste(s) gilets jaunes",
    "LR" : "Les Républicains",
    "MDM" : "MoDem",
    "NC" : "Liste(s) non classées",
    "RDG" : "Parti radical de gauche",
    "REG" : "Liste(s) régionaliste(s)",
    "REM" : "La République en marche",
    "RN" : "Rassemblement national",
    "SOC" : "Parti socialiste",
    "UC" : "Union du centre",
    "UD" : "Union de la droite",
    "UDI" : "UDI",
    "UG" : "Union de la gauche",
    "VEC" : "Europe Ecologie Les Verts"
})

In [25]:
mapping_couleurs = {
    "Extrême gauche" : "#de2c35",
    "Gauche" : "#ff2dff",
    "Centre" : "#e7a60e",
    "Droite": "#2f2fff",
    "Extrême droite" : "#000000",
    "Divers ou non classé" : "#8c8c8c"
}

In [26]:
resultats_communes_2020_bloc_et_nuance["Couleur"] = resultats_communes_2020_bloc_et_nuance["Bloc dominant"].map(mapping_couleurs)

In [ ]:
resultats_communes_2020_bloc_et_nuance.to_excel("../data/clean/resultats_2020_clean.xlsx")

In [27]:
df_base_2026 = pd.read_csv("../data/raw/resultats_municipales_2026.csv", encoding="utf-8-sig", sep=";", low_memory=False)

In [28]:
df_base_2026 = df_base_2026.rename(columns={"Code commune": "code_insee"})

df_base_2026["code_insee"] = df_base_2026["code_insee"].astype(str).str.zfill(5)

In [29]:
df_base_2026_regions = pd.merge(df_base_2026, df_info_region, on="code_insee", how="inner")

In [30]:
resultats = []

for index, row in df_base_2026_regions.iterrows():
    commune = row["Libellé commune"]
    code_insee = row["code_insee"]
    region = row["Nom de la région"]
    participation = row["% Votants"]

    for n in range (1,14):
        nuance = row.iloc[22 + (n-1) * 13]
        pourcentage_voix = row.iloc[27 + (n-1) * 13]

        if pd.notna(nuance):
            resultats.append({
                "commune": commune,
                "code_insee": code_insee,
                "region" : region,
                "participation" : participation,
                "nuance" : nuance,
                "pourcentage_voix" : pourcentage_voix,
            })

df_long_2026_toutes_regions = pd.DataFrame(resultats)

In [31]:
df_long_2026_toutes_regions["nuance"] = df_long_2026_toutes_regions["nuance"].str[1:]

In [32]:
mapping_politique_2026 = {
    # extrême-gauche
    "EXG" : "Extrême gauche",

    # gauche
    "COM" : "Gauche", "FI" : "Gauche", "GEN" : "Gauche", "SOC" : "Gauche", "PLP" : "Gauche", "RDG" : "Gauche", "DVG" : "Gauche", "UG" : "Gauche", "VEC" : "Gauche",

    # centre
    "REN" :  "Centre", "MDM" : "Centre", "UDI" : "Centre", "UC" : "Centre", "DVC" : "Centre", "HOR" : "Centre", "PR" : "Centre",

    # droite
    "LR" : "Droite", "UD" : "Droite", "DVD" : "Droite", "DSV" : "Droite",

    # extrême-droite
    "RN" : "Extrême droite", "EXD" : "Extrême droite", "UDR" : "Extrême droite", "UXD" : "Extrême droite", "REC" : "Extrême droite",

    # divers ou non classé
    "ECO" : "Divers ou non classé", "DIV" : "Divers ou non classé", "REG" : "Divers ou non classé", "" : "Divers ou non classé", "NC" : "Divers ou non classé", "ANM" : "Divers ou non classé"
}

In [33]:
df_long_2026_toutes_regions["bloc"] = df_long_2026_toutes_regions["nuance"].map(mapping_politique_2026)

In [34]:
df_long_2026_toutes_regions["pourcentage_voix"] = df_long_2026_toutes_regions["pourcentage_voix"].str.replace(",",".")
df_long_2026_toutes_regions["pourcentage_voix"] = df_long_2026_toutes_regions["pourcentage_voix"].str.replace("%","")
df_long_2026_toutes_regions["pourcentage_voix"] = pd.to_numeric(df_long_2026_toutes_regions["pourcentage_voix"])

In [35]:
df_long_2026_toutes_regions["participation"] = df_long_2026_toutes_regions["participation"].str.replace("%","")

In [36]:
resultats_commune_bloc_2026 = df_long_2026_toutes_regions.groupby(["commune", "code_insee", "region", "participation", "bloc"])["pourcentage_voix"].sum().reset_index()

In [37]:
resultats_commune_bloc_2026_wide = resultats_commune_bloc_2026.pivot_table(
    index=["commune", "code_insee", "region", "participation"], 
    columns="bloc", 
    values="pourcentage_voix"
).reset_index()

In [38]:
resultats_commune_bloc_2026_wide = resultats_commune_bloc_2026_wide[["commune", "code_insee", "region", "participation", "Extrême gauche", "Gauche", "Centre", "Droite", "Extrême droite", "Divers ou non classé"]]

In [39]:
resultats_commune_bloc_2026_wide["Bloc dominant"] = resultats_commune_bloc_2026_wide[["Extrême gauche", "Gauche", "Centre", "Droite", "Extrême droite", "Divers ou non classé"]].idxmax(axis=1)

In [40]:
resultats_commune_2026_nuance = df_long_2026_toutes_regions.groupby(["code_insee", "nuance"])["pourcentage_voix"].sum().reset_index()

In [41]:
resultats_commune_2026_nuance_wide = resultats_commune_2026_nuance.pivot_table(
    index=["code_insee"], 
    columns="nuance", 
    values="pourcentage_voix"
).reset_index()

In [42]:
resultats_communes_2026_bloc_et_nuance= pd.merge(resultats_commune_bloc_2026_wide, resultats_commune_2026_nuance_wide, on="code_insee", how="inner")

In [43]:
resultats_communes_2026_bloc_et_nuance = resultats_communes_2026_bloc_et_nuance[["commune", "code_insee", "region", "participation", "Bloc dominant", "Extrême gauche", "Gauche", "Centre", "Droite", "Extrême droite", "Divers ou non classé", "EXG", "FI", "COM", "SOC", "VEC", "UG", "DVG", "REN", "MDM", "HOR", "UDI", "UC", "DVC", "ECO", "DIV", "REG", "LR", "UD", "DVD", "DSV", "UDR", "RN", "REC", "EXD", "UXD"]]

In [44]:
resultats_communes_2026_bloc_et_nuance = resultats_communes_2026_bloc_et_nuance.rename(columns={
    "COM" : "Parti communiste français",
    "DIV" : "Liste(s) divers",
    "DSV" : "Droite souverainiste",
    "DVC" : "Liste(s) divers centre",
    "DVD" : "Liste(s) divers droite",
    "DVG" : "Liste(s) divers gauche",
    "ECO" : "Autre(s) liste(s) écologiste(s)",
    "EXD" : "Liste(s) d'extrême droite",
    "EXG" : "Liste(s) d'extrême gauche",
    "FI" : "La France insoumise",
    "HOR": "Horizons",
    "LR" : "Les Républicains",
    "MDM" : "MoDem",
    "REC" : "Reconquête",
    "REG" : "Liste(s) régionaliste(s)",
    "REN" : "Renaissance",
    "RN" : "Rassemblement national",
    "SOC" : "Parti socialiste",
    "UC" : "Union du centre",
    "UD" : "Union de la droite",
    "UDI" : "UDI",
    "UDR": "Union des droites pour la République",
    "UG" : "Union de la gauche",
    "UXD": "Union de l'extrême droite",
    "VEC" : "Europe Ecologie Les Verts"
})

In [45]:
resultats_communes_2026_bloc_et_nuance["Couleur"] = resultats_communes_2026_bloc_et_nuance["Bloc dominant"].map(mapping_couleurs)

In [46]:
df_comparaison = pd.merge(
    resultats_communes_2020_bloc_et_nuance[["code_insee", "Bloc dominant", "Extrême gauche", "Gauche", "Centre", "Droite", "Extrême droite", "Divers ou non classé"]],
    resultats_communes_2026_bloc_et_nuance[["code_insee", "Bloc dominant", "Extrême gauche", "Gauche", "Centre", "Droite", "Extrême droite", "Divers ou non classé"]],
    on="code_insee",
    suffixes=("_2020", "_2026")
)

In [47]:
blocs = ["Extrême gauche", "Gauche", "Centre", "Droite", "Extrême droite", "Divers ou non classé"]

for bloc in blocs:
    df_comparaison[f"Evolution {bloc}"] = df_comparaison[f"{bloc}_2026"].fillna(0) - df_comparaison[f"{bloc}_2020"].fillna(0)

In [48]:
for bloc in blocs:
    df_comparaison[f"Signe {bloc}"] = np.where(df_comparaison[f"Evolution {bloc}"] > 0, "+",
                                      np.where(df_comparaison[f"Evolution {bloc}"] < 0, "-", ""))
    df_comparaison[f"Couleur signe {bloc}"] = df_comparaison[f"Signe {bloc}"].map({"+": "#00aa00", "-": "#ff0000", "": "#000000"})

In [49]:
for bloc in blocs:
    df_comparaison[f"Evolution {bloc}"] = df_comparaison[f"Evolution {bloc}"].abs()

In [50]:
colonnes_a_merger = ["code_insee"] + \
                    [f"Evolution {bloc}" for bloc in blocs] + \
                    [f"Signe {bloc}" for bloc in blocs] + \
                    [f"Couleur signe {bloc}" for bloc in blocs]

resultats_communes_2026_bloc_et_nuance = pd.merge(
    resultats_communes_2026_bloc_et_nuance,
    df_comparaison[colonnes_a_merger],
    on="code_insee",
    how="left"
)

In [ ]:
resultats_communes_2026_bloc_et_nuance.to_excel("../data/clean/resultats_2026_clean.xlsx")

In [ ]:
regions = ["Auvergne-Rhône-Alpes", "Bourgogne-Franche-Comté", "Bretagne", "Centre-Val de Loire", "Grand Est", "Hauts-de-France", "Île-de-France", "Normandie", "Nouvelle-Aquitaine", "Occitanie", "Pays de la Loire", "Provence-Alpes-Côte d'Azur"]

In [ ]:
for region in regions:
    df_region_2020 = resultats_communes_2020_bloc_et_nuance[resultats_communes_2020_bloc_et_nuance["region"] == region]
    df_region_2020.to_excel(f"../data/clean/resultats_{region}_2020.xlsx", index=False)

In [ ]:
for region in regions:
    df_region_2026 = resultats_communes_2026_bloc_et_nuance[resultats_communes_2026_bloc_et_nuance["region"] == region]
    df_region_2026.to_excel(f"../data/clean/resultats_{region}_2026.xlsx", index=False)